# Expresiones regulares en Python

Las expresiones regulares permiten buscar, extraer, validar y sustituir patrones no triviales de texto.

En este cuaderno utilizaremos el módulo `re` de Python.


In [ ]:
import re
from datetime import datetime

##  El módulo `re`

Es recomendable utilizar cadenas *raw* (`r"..."`) para evitar problemas con las barras invertidas.


In [ ]:
texto = "El pedido AB-1234 fue enviado el 12/09/2026"

patron = re.compile(r"\d+")
re.findall(patron, texto)

`re.compile()` convierte (o "compila") una expresión regular en texto plano a un objeto de patrón de expresión regular (Pattern Object). Ese objeto resultante ya tiene procesada la lógica de la búsqueda, por lo que contiene sus propios métodos para buscar coincidencias (como` .search()`, `.match()`, `.findall()`, `.sub()`, etc.) sin necesidad de pasarle el patrón de texto cada vez.

## Elementos básicos

| Elemento | Significado |
|---|---|
| `.` | Cualquier carácter |
| `^` | Inicio del texto o línea |
| `$` | Final del texto o línea |
| `\d` | Dígito |
| `\w` | Letra, dígito o guion bajo |
| `\s` | Coincide con un  espacio en blanco o carácter espaciador: \t\n\r\f\v. |
| `\S` |Coincide con cualquier carácter que no sea espacio en blanco|
| `[abc]` | Uno de los caracteres indicados |
| `[^abc]` | Cualquier carácter excepto los indicados |
| `[a-z]` | Rango de caracteres |


In [ ]:
texto = "Python 3.12"
patron = re.compile(r"[A-Za-z]+")

re.findall(patron, texto)

## Funciones principales que aporta re.compile()

Al compilar un patrón, es posible ejecutar las búsquedas directamente sobre la variable resultante:

* **patron.search(texto)**: Busca la primera coincidencia en cualquier parte del texto.

* **patron.match(texto)**: Busca si el texto comienza haciendo coincidencia con el patrón.

* **patron.findall(texto)**: Devuelve una lista con todas las coincidencias encontradas.

* **patron.finditer(texto)**: Devuelve un iterador de objetos Match (ideal para ahorrar memoria en textos gigantes).

* **patron.sub(reemplazo, texto)**: Reemplaza las coincidencias encontradas por otro texto.

In [ ]:
texto = "Python 3.12"
patron = re.compile(r"[A-Za-z]+")

#re.findall(patron, texto)
patron.findall(texto)

##  Cuantificadores

| Cuantificador | Significado |
|---|---|
| `*` | Cero o más veces |
| `+` | Una o más veces |
| `?` | Cero o una vez |
| `{n}` | Exactamente `n` veces |
| `{n,}` | Al menos `n` veces |
| `{n,m}` | Entre `n` y `m` veces |


In [ ]:
texto = "abc ab abbbb"

patrones = {
    "una o más b": r"ab+",
    "entre 1 y 3 b": r"ab{1,3}",
    "exactamente 2 b": r"ab{2}"
}

for descripcion, patron in patrones.items():
    print(descripcion, re.compile(patron).findall(texto))

## Búsqueda de patrones

- `search()`: busca la primera coincidencia en cualquier posición.
- `match()`: busca desde el principio.
- `fullmatch()`: exige que coincida todo el texto.


In [ ]:
patron = re.compile(r"\d{4}")

for texto in ["Código: 1234", "1234", "12345"]:
    print(f"Texto: {texto}\n  search: {patron.search(texto)}\n  match: {patron.match(texto)} \n  fullmatch: {patron.fullmatch(texto)}"
    )

##  Grupos y extracción de información

Los paréntesis permiten agrupar y capturar partes de una coincidencia.


__Extracción de IP__

El patrón `r'^(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})'` busca cuatro grupos de 1 a 3 dígitos separados por puntos justo al inicio de la cadena.

In [ ]:
texto_log = """213.180.203.109 - - [15/Sep/2023:00:12:18 +0200] 
           "GET /robots.txt HTTP/1.1" 302 567 "-" "Mozilla/5.0 (compatible;
            YandexBot/3.0; +http://yandex.com/bots)"\n"""
 
# Patrón para capturar la IP al inicio de la línea
patron = re.compile(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}")

# Buscar la coincidencia
resultado = re.search(patron, texto_log)  
if resultado:
    print(resultado.group())

* `^`: Asegura que la coincidencia comience justo al inicio del texto.
* `\d{1,3}`: Coincide con un dígito numérico (0-9) que se repita entre 1 y 3 veces.
* `\.`: Escapa el punto (ya que en regex el punto solo . significa "cualquier carácter"). Esto fuerza a buscar un punto literal.

__Busqueda de fechas__

In [ ]:
# Compilamos el patrón una sola vez 
patron_fecha = re.compile(r"\[.*\]")

# Buscar la coincidencia

#resultado = re.search(patron_fecha, line)  
resultado_search = patron_fecha.search(texto_log)
resultado_match = patron_fecha.match(texto_log)
if resultado:
    print(resultado_search)
    print(resultado_match)

Modificamos la ER para poder extraer fecha

In [ ]:
datere = re.compile(r'\[(\d+).*\]')
m = datere.search(texto_log)
print(f'{texto} \n{m.group()}, \n{m.group(0)} ')


Extraemos las distintas partes de la fecha:

In [ ]:
datere = re.compile(r'\[(\d+)/(\w+)/(\d+):(\d+):(\d+):(\d+)\s*\+(\d+)\]')
m = datere.search(texto_log)
m.groups(), m.group(0), m.group(1), m.group(2)

Todo este esfuerzo ha estado bien para ir entendiendo los componentes de las expresiones regulares. Pero para analizar las fechas es mejor usar la función [`strptime` de la clase `datetime`](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior)

In [ ]:
from datetime import datetime
datere = re.compile(r'\[(.+)\]')
m = datere.search(texto_log)
ts = datetime.strptime(m.group(1), '%d/%b/%Y:%H:%M:%S %z')
ts


## Grupos con nombre y grupos de no captura

El símbolo `? ` combinado con paréntesis `(?...)`, cambia la forma en que funciona un grupo de captura:

* **(?:...)** - Grupo de no captura: Agrupa caracteres sin guardarlos en la memoria de grupos (por ejemplo, `(?:ab)+`).

* **(?P<nombre>...)** - Grupo nombrado: Asigna un nombre al grupo capturado para acceder a él mediante `match.group('nombre')`.


__Ejemplo:__

In [ ]:
texto = "Producto: AB-1234"

patron = re.compile(
    r"Producto: (?P<familia>[A-Z]{2})-(?P<numero>\d{4})"
)

resultado = patron.search(texto)
resultado.groupdict()

__Ejemplo:__

In [ ]:
datere = re.compile(r'\[(?P<date>.+)\]\s+"(?P<command>\w+)\s+(?P<page>\S+)\s+(?P<protocol>\S+)"')
m = datere.search(texto_log)
ts = datetime.strptime(m.group('date'), '%d/%b/%Y:%H:%M:%S %z')
ts, m.group('command'), m.group('page'), m.group('protocol')

__Ejemplo:__

In [ ]:
patron = re.compile(r"(?:https?://)?(\w+\.com)")
patron.search("Visita https://ejemplo.com").groups()

## Opciones o *flags*

Las opciones disponibles en el módulo re de Python (también llamadas **Banderas** o **Flags**) se utilizan para modificar el comportamiento predeterminado de las expresiones regulares al compilar patrones con `re.compile()` o al usar funciones como `re.search()`

| Opción (Larga) | Abreviatura  | Descripción |
| :--- | :---: |  :--- |
| `re.IGNORECASE` | `re.I` |  Ignora la diferencia entre mayúsculas y minúsculas al buscar. |
| `re.MULTILINE` | `re.M` |  Hace que `^` y `$` coincidan al inicio y final de cada *línea*, no solo de todo el texto. |
| `re.DOTALL` | `re.S` |  Permite que el carácter punto `.` coincida con **cualquier** carácter, incluidos los saltos de línea (`\n`). |
| `re.VERBOSE` | `re.X` |  Permite escribir expresiones regulares multilínea, ignorando espacios en blanco y permitiendo comentarios (`#`). |
| `re.ASCII` | `re.A` |  Fuerza a las secuencias `\w`, `\W`, `\b`, `\B`, `\d`, `\D`, `\s` y `\S` a coincidir solo con caracteres ASCII. |
| `re.UNICODE` | `re.U` |  Hace que las secuencias de escape sigan las propiedades de caracteres Unicode (activado por defecto en Python 3). |
| `re.LOCALE` | `re.L` | Hace que `\w`, `\W`, `\b`, `\B` dependan de la configuración regional (*locale*) actual (obsoleto/no recomendado). |
| `re.NOFLAG` | N/A |  Representa la ausencia de banderas (equivale a `0`). Útil para indicar que no se aplican opciones. |


In [ ]:
texto = "Python PYTHON python"
patron = re.compile(r"python", re.IGNORECASE)
re.findall(patron, texto), patron.findall(texto)

__Ejemplo con `re.VERBOSE`__

In [ ]:
patron = re.compile(r'''
    (?P<usuario>\w+)       # Usuario
    @
    (?P<dominio>[\w.-]+)   # Dominio
    \. 
    (?P<extension>[a-z]{2,})
''', re.VERBOSE | re.IGNORECASE)

patron.search("correo: alumno@universidad.es").groupdict()

__Ejemplo con re.ASCII__

para considerar solo caracteres ASCII.

In [ ]:
p1 = re.compile(r'\w+')
p2 = re.compile(r'\w+', re.ASCII)
m1 = p1.match('avión')
m2 = p2.match('avión')
m1, m2

__Ejemplo con `rs.DOTALL`__

La bandera `re.DOTALL` le indica que el punto `.` debe incluir también los saltos de línea (\n).

In [ ]:
s = 'abcá\ncñde'
p1 = re.compile('.*')
p2 = re.compile('.*', re.DOTALL)
m1 = p1.match(s)
m2 = p2.match(s)
m1, m2

##  Funciones principales de `re`

In [ ]:
texto = "Python, Java, Python, R"

print(re.findall(r"\w+", texto))
print(list(re.finditer(r"Python", texto)))
print(re.split(r",\s*", texto))
print(re.sub(r"Python", "PYTHON", texto))

## Validación de datos

Las expresiones regulares pueden comprobar el formato de un dato, aunque no siempre garantizan que el valor sea semánticamente correcto.


In [ ]:
patrones = {
    "correo": r"^[\w.-]+@[\w.-]+\.[A-Za-z]{2,}$",
    "código postal": r"^\d{5}$",
    "teléfono": r"^\d{9}$",
    "fecha": r"^\d{2}/\d{2}/\d{4}$"
}

datos = {
    "correo": "alumno@universidad.es",
    "código postal": "28001",
    "teléfono": "612345678",
    "fecha": "14/09/2026"
}

for tipo, valor in datos.items():
    valido = re.fullmatch(re.compile(patrones[tipo]), valor) is not None
    print(tipo, valido)

##  Buenas prácticas

- Utilizar cadenas *raw*: `r"\d+"`.
- Compilar los patrones que se reutilizan.
- Preferir grupos con nombre en patrones complejos.
- Documentar las expresiones difíciles.
- Usar `re.VERBOSE` cuando el patrón sea largo.
- No utilizar expresiones regulares cuando exista una solución más clara.
- Validar siempre los resultados antes de usar `group()`.


## Referencias


* Documentación del módulo re [https://docs.python.org/3/howto/regex.html#matching-characters](https://docs.python.org/3/howto/regex.html#matching-characters)

*  Documentación oficial [https://docs.python.org/3/howto/regex.html#regular-expression-howto](https://docs.python.org/3/howto/regex.html#regular-expression-howto)

---
